In [4]:
import pandas as pd
import glob

files = glob.glob("sap-o2c-data/billing_document_items/*.jsonl")

df_list = [pd.read_json(f, lines=True) for f in files]

df = pd.concat(df_list, ignore_index=True)

print(df.shape)
print(df.head())

(245, 9)
   billingDocument  billingDocumentItem        material  billingQuantity  \
0         90504298                   10  B8907367041603                1   
1         90678703                   10  B8907367032571                1   
2         90678698                   10  B8907367022152                1   
3         90504301                   10  B8907367039570                1   
4         90678702                   10  S8907367008620                1   

  billingQuantityUnit  netAmount transactionCurrency  referenceSdDocument  \
0                  PC     533.05                 INR             80738109   
1                  PC     305.90                 INR             80754606   
2                  PC     382.90                 INR             80754601   
3                  PC    1091.53                 INR             80738111   
4                  PC     299.00                 INR             80754605   

   referenceSdDocumentItem  
0                       10  
1            

In [6]:
files = glob.glob("sap-o2c-data/billing_document_items/*.jsonl")
print(files)
df_items = pd.concat(
    [pd.read_json(f, lines=True) for f in files],
    ignore_index=True
)

print(df_items.columns)

['sap-o2c-data/billing_document_items\\part-20251119-133432-233.jsonl', 'sap-o2c-data/billing_document_items\\part-20251119-133432-978.jsonl']
Index(['billingDocument', 'billingDocumentItem', 'material', 'billingQuantity',
       'billingQuantityUnit', 'netAmount', 'transactionCurrency',
       'referenceSdDocument', 'referenceSdDocumentItem'],
      dtype='object')


In [7]:
df_items = df_items.rename(columns={
    "billingDocument": "invoice_id",
    "material": "product_id",
    "billingQuantity": "quantity",
    "netAmount": "item_amount",
    "referenceSdDocument": "delivery_id"
})

In [8]:
df_items = df_items[[
    "invoice_id",
    "product_id",
    "quantity",
    "item_amount",
    "delivery_id"
]]

In [10]:
import sqlite3

conn = sqlite3.connect("data.db")
df_items.to_sql("invoice_items", conn, if_exists="replace", index=False)

245

In [11]:
query = """
SELECT i.invoice_id, i.customer_id, ii.product_id, ii.delivery_id
FROM invoices i
JOIN invoice_items ii
ON i.invoice_id = ii.invoice_id
LIMIT 5;
"""

print(pd.read_sql(query, conn))

   invoice_id  customer_id      product_id  delivery_id
0    90504248    320000083  S8907367001003     80738072
1    90628265    320000082  S8907367013532     80754575
2    90628266    320000082  S8907367002512     80754576
3    90504274    320000083  S8907367003380     80738091
4    90504242    320000083  S8907367025344     80738068


In [13]:
query = """
SELECT product_id, COUNT(*) as total_bills
FROM invoice_items
GROUP BY product_id
ORDER BY total_bills DESC
LIMIT 5;
"""

print(pd.read_sql(query, conn))

       product_id  total_bills
0  S8907367039280           22
1  S8907367008620           22
2  S8907367042006           16
3  S8907367013532           15
4  S8907367001003            9
